# EDA notebook - Group 22

This notebook was made by:

- Alexandre Batista: 20250419
- Daniel Caridade: 20211588
- Luis Mendes: 20221949
- Mehmet Karaca: 20250344
- Veronica Mendes: 20221945



# 1. Notebook objective

Same random text



# 2. Library importation & Spark Session

__`Step 1`__ - Installing Java 17 as it is required for Spark. <br>

Uncomment the cells below if Java 17 is not yet installed in your environment.
For the rest of the packages and dependencies, uv is being used, so the required steps are to install uv in your environment and then run the following code: `uv sync` to get all the depencies and the libraries installed in your environment.

In [2]:
# Install Java 17 (Required for Spark)
#!sudo apt-get update
#!sudo apt-get install -y openjdk-17-jdk-headless
#!java -version

__`Step 2`__ - Importing the required libraries.

In [15]:
import os
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import NumericType, StringType

__`Step 3`__ - Creating the Spark Session. <br>

In the Spark Session some modifications were done compared to the code showed in class, which are:

- `local[*]`: Runs Spark in local mode using all available CPU cores. 
- `spark.driver.memory", "4g`: The driver memory was chosen to 4g to ensure enough memory for driver processes.
- `spark.sql.shuffle.partitions", "30"`: Reduces the default number of shuffle partitions (which is often 200). This was a design choice as EDA usually require a lot of shuffle operations in the analysis of the dataset, so by reducing this shuffle partitions we improve the performance of the code as our dataset are not huge. 
- `spark.sparkContext.setLogLevel("WARN")`: Surpress verbose INFO logs, to make the notebook cleaner.

In [4]:
# Create Spark Session
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("Online_Retail_EDA") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "4g") \
        .config("spark.sql.shuffle.partitions", "100") \
        .getOrCreate()

# Reduce Spark logging noise
spark.sparkContext.setLogLevel("WARN")

print("Spark Session initialized successfully!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 19:14:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session initialized successfully!


# 3. Data integration

__`Step 4`__ - Creating two parquet files out of the `online_retail_II.xlsx` as this file has two sheets one with transctional data of 2009 and another with transactional data of 2010. <br>

__Important note:__ Notice that the `Pandas` library was used to import that data into the notebook, instead of a Spark Native way, as Spark does not possesses this native way of importing a `.xlsx` file like it does for `.csv` files. Nonetheless, testing with Spark Excel connector were tried, but after getting some issues with the Spark Session due to this connector this approach was considered more valuable, as it will only be used for 1 step for getting the data into a parquet and all the next steps, like importing the data and concatenating the dataset will be done through Spark native options as can be seen in the steps that come after this one.

In [5]:
# Loading the two datasets as pandas dataframes
df_0910 = pd.read_excel("../Data/online_retail_II.xlsx", sheet_name="Year 2009-2010")

df_1011 = pd.read_excel("../Data/online_retail_II.xlsx", sheet_name="Year 2010-2011")

# Forcing object types to be of type string to avoid errors when converting to parquet
for df in [df_0910, df_1011]:
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str)

# Write each of the pandas datasframes as a parquet file
df_0910.to_parquet("../Data/online_retail_0910.parquet", index=False)
df_1011.to_parquet("../Data/online_retail_1011.parquet", index=False)

__`Step 5`__ - Importing the parquet files into Spark as dataframes. 

In [6]:
# Loading the files into Spark

df_spark_0910 = spark.read.parquet("../Data/online_retail_0910.parquet")
df_spark_1011 = spark.read.parquet("../Data/online_retail_1011.parquet")

# Printing their schema for looking into any possible issues
df_spark_0910.printSchema()
df_spark_1011.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



__`Step 6`__ - Concatenating both datasets and saving it new parket file that can be loaded directly in other notebooks. <br>

__Note:__ For this notebook we will keep using `df` created in the cell of code below as it contains the necessary information for the exploratory data analysis.

In [7]:
# Combining the sales transcations from 2009 with the ones from 2010
df = df_spark_0910.unionByName(df_spark_1011)

# Saving this result for future uses on other notebooks
df.write.mode("overwrite").parquet("../Data/online_retail_II.parquet")

# 4. Data Exploration

## 4.1. Data Content

__`Step 7`__ - Printing the schema and the 5 five rows of the data.

In [8]:
df.printSchema()
df.show(5)

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)

+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|RECORD FRAME 7" S...|      

From this output the aspect to conclude is the fact that the variable `Invoice` is stored as a string, dispite only displaying numbers in the first 5 rows, indicating that there is a need for further analysis on this subject.

__`Step 8`__ - Checking the number of rows and the number of columns.

In [9]:
df.count(), len(df.columns)

(1067371, 8)

The dataframe contains a grand total of 1067371 rows and 8 columns, indicating that the big dimensionality on the data is presented on the rows, while we have very few columns in the data.

__`Step 9`__ - Checking the number of missing values per variable.

In [10]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c)for c in df.columns]).show()

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|      0|        0|       4382|       0|          0|    0|     243007|      0|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



The only columns with missing values are the columns `Description` which has a small fraction of missing values, only **0.41%** and `Customer ID` with **22.77%**, that need to be dealt with in Data Preprocesssing.

### Data description

| Column Name   | Data Type | Description                                  |
|---------------|-----------|----------------------------------------------|
| Invoice       | String    | Unique invoice number for each transaction   |
| StockCode     | String    | Product code identifying each item           |
| Description   | String    | Name/description of the product              |
| Quantity      | Long      | Number of units purchased in the transaction |
| InvoiceDate   | Timestamp | Date and time when the invoice was created   |
| Price         | Double    | Unit price of the product                    |
| Customer ID   | Double    | Unique identifier for the customer           |
| Country       | String    | Country where the customer is located        |

## 4.2. Descriptive statistics

__`Step 10`__ - Checking some descriptive statistics of the numerical variables.

In [11]:
# Identifying the numerical columns in the dataset
numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]

# Checking some descriptive statistics for the numerical columns
df.select(numeric_cols).describe().show()

+-------+------------------+------------------+------------------+
|summary|          Quantity|             Price|       Customer ID|
+-------+------------------+------------------+------------------+
|  count|           1067371|           1067371|            824364|
|   mean|   9.9388984711033|  4.64938772741356| 15324.63850435002|
| stddev|172.70579407675336|123.55305872146316|1697.4644503793113|
|    min|            -80995|         -53594.36|           12346.0|
|    max|             80995|           38970.0|           18287.0|
+-------+------------------+------------------+------------------+



__`Step 11`__ - Checking the number of unique values in the per each column in the dataset.

In [12]:
df.agg(*[F.countDistinct(F.col(c)).alias(c) for c in df.columns]).show()

+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|Invoice|StockCode|Description|Quantity|InvoiceDate|Price|Customer ID|Country|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+
|  53628|     5305|       5698|    1057|      47635| 2807|       5942|     43|
+-------+---------+-----------+--------+-----------+-----+-----------+-------+



__`Step 12`__ - Checking the first transaction date and the last transaction dates.

In [13]:
df.agg(
    F.min("InvoiceDate").alias("First transaction date"),
    F.max("InvoiceDate").alias("Last transation date")
).show()

+----------------------+--------------------+
|First transaction date|Last transation date|
+----------------------+--------------------+
|   2009-12-01 07:45:00| 2011-12-09 12:50:00|
+----------------------+--------------------+



From this analysis we can conclude the following:

1. The company has sales that ahve negative prices and negative quantities which at a first glance can represent two things: **(1) they represent incoherences in the data that need to be dealt with** or **(2) they represent cancelled orders, which would justify this behaviour.**
2. The average number of products bought by the customer are 10 products. Moreover, the maximum number of products that were bought by a customer were 80995 sales making it very clear the presence of outliers in the variables.
3. The average price of the products in the dataset is **4.65€**, indicating that the store sales fairly cheap products. Similarly to the quantity the top price of a product is 38970 indicating not just the presence of outliers, but also the fact that the store has a reasonable range of different product pricing, with the majority being fairly cheap and some other having a premium price tag.
4. The dataset contains in total **53628 invoices**, selling **5698** different products to **5942** different customers in **43 countries**.
5. The data spans from the first day of December of 2009 until the night day of December of 2011, presenting data from 2 years.

## 4.3. Missing values analysis

__`Step 13`__ - Checking the there are any values expressed as NA or any other type of missing representation that are not classified as missing values.

In [19]:
# Creating a list with all the categorical variables
string_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]

# Creating a list with common textual missing-value representations
na_values = ["NA", "N/A", "NULL", "null", "NaN", "nan", "", " "]

# Count the occurrences per column
df.select([F.count(F.when(F.trim(F.col(c)).isin(na_values), c)).alias(c) for c in string_columns]).show()

+-------+---------+-----------+-------+
|Invoice|StockCode|Description|Country|
+-------+---------+-----------+-------+
|      0|        0|          0|      0|
+-------+---------+-----------+-------+



__`Step 14`__ - Checking if the missing customer ID are related to any specific country.

In [28]:
from pyspark.sql import functions as F

# Total transactions per country
total_transactions = (
    df.groupBy("Country")
      .agg(
          F.countDistinct("Invoice").alias("total_transactions")
      )
)

# Transactions with missing Customer ID per country
missing_customer_transactions = (
    df.filter(F.col("Customer ID").isNull())
      .groupBy("Country")
      .agg(
          F.countDistinct("Invoice").alias("missing_customer_transactions")
      )
)

# Keep only countries with missing values
missing_customer_analysis = (
    missing_customer_transactions
    .join(
        total_transactions,
        on="Country",
        how="inner"
    )
    .withColumn(
        "missing_customer_pct",
        F.round(
            (
                F.col("missing_customer_transactions") /
                F.col("total_transactions")
            ) * 100,
            2
        )
    )
    .orderBy(F.desc("missing_customer_pct"))
)

missing_customer_analysis.show(truncate=False)

+--------------------+-----------------------------+------------------+--------------------+
|Country             |missing_customer_transactions|total_transactions|missing_customer_pct|
+--------------------+-----------------------------+------------------+--------------------+
|Hong Kong           |21                           |21                |100.0               |
|Bermuda             |1                            |1                 |100.0               |
|Lebanon             |2                            |3                 |66.67               |
|Bahrain             |8                            |12                |66.67               |
|RSA                 |3                            |5                 |60.0                |
|United Arab Emirates|8                            |22                |36.36               |
|Nigeria             |1                            |3                 |33.33               |
|Unspecified         |9                            |28                

Based on the above table we can conclude that the majority of the missing `Customer ID` are from customers from the `UK`. Nonetheless, they only **17.52%** of the transactions for that country. <br>
In countries such as `Hong Kong` and `Bermuda` all the transactions have a missing `Customer ID`, while `Sweden` is the country in which less transaction don't have a specified ID with only **0.78%**. <br>
Additionally some transactions have an Unspecified country, currenponding to **28 transactions** in which **9** have a missing `Customer ID`. <br>
This missing values need to be carefully handled as customers can make purchases in the E-Commerce platform as guests, meaning they don0t need to have a `Customer ID` which would explain why this missing values exist.

__`Step 15`__ - Checking what is the percentage of the total revenue that the missing `Customer ID` transactions represent.

In [27]:
# Creating a column called Revenue which represent the Price of the product times the quantity
df = df.withColumn(
    "Revenue",
    F.col("Quantity") * F.col("Price")
)

# Compute revenue per group in one pass
revenue_by_status = (
    df.groupBy(
        F.when(F.col("Customer ID").isNull(), "Missing")
         .otherwise("Present")
         .alias("customer_id_status")
    )
    .agg(
        F.sum(F.col("Quantity") * F.col("Price")).alias("total_revenue")
    )
)

# Total revenue (single-row aggregate)
total_revenue = revenue_by_status.agg(
    F.sum("total_revenue").alias("global_revenue")
)

# Compute the percentage
result = (
    revenue_by_status
    .crossJoin(total_revenue)
    .withColumn(
        "revenue_pct",
        F.round(
            F.col("total_revenue") / F.col("global_revenue") * 100,
            2
        )
    )
    .select("customer_id_status", "revenue_pct")
    .orderBy(F.desc("revenue_pct"))
)

result.show()

+------------------+-----------+
|customer_id_status|revenue_pct|
+------------------+-----------+
|           Present|      86.32|
|           Missing|      13.68|
+------------------+-----------+



The transactions in which the there is a missing ID represent fairly **14%** of the total revenue done in the business, showcasing that it is relevant to understand this customers, as they represent an important portion of the companies revenue. Completly disregarding this customers completely, would lead to putting **14%** of the companies sales revenue on a gamble that these customers won't leave the company in a near future which is not very plausible without any intervention or motivation for them to keep buying in the company.

__`Step 16`__ - Checking how many transactions witout  `Customer ID` represent cancellations. Notice that cancellation records are the ones that start with `C` based on the metadata on the website where the data was obtained.

In [41]:
df.filter(F.col("Customer ID").isNull()) \
  .groupBy(
      F.when(
          F.col("Invoice").startswith("C"),
          "Return"
      ).otherwise("Purchase").alias("transaction_type")
  ) \
  .agg(
      F.countDistinct("Invoice").alias("num_transactions")
  ) \
  .show()

+----------------+----------------+
|transaction_type|num_transactions|
+----------------+----------------+
|        Purchase|            8361|
|          Return|             391|
+----------------+----------------+



We have a grand total of **391 transactions** that represent cancellations, nonetheless the majority of them **8361 transactions** represent actual purchases on the data. 

__`Step 17`__ - Checking if the linked descriptions are linked to specific products.

In [31]:
df.filter(F.col("Description").isNull()) \
      .groupBy("StockCode") \
      .count() \
      .orderBy(F.desc("count")).show()

+---------+-----+
|StockCode|count|
+---------+-----+
|    22139|   12|
|    84990|   12|
|    35965|   11|
|    79321|   11|
|    22950|   10|
|    23084|   10|
|    22087|    9|
|    22084|    9|
|    37461|    8|
|    71477|    8|
|    84977|    7|
|    35970|    7|
|   72803B|    7|
|     POST|    7|
|   84795D|    7|
|    22451|    7|
|    21768|    7|
|    22528|    7|
|    37509|    7|
|    21690|    6|
+---------+-----+
only showing top 20 rows



__`Step 17.1`__ Checking if those `StokeCode` have a `Description` in another row, so it is an easy way to impute this values in ETL.

In [32]:
summary = (
    df.groupBy("StockCode")
      .agg(
          F.count(F.when(F.col("Description").isNull(), True)).alias("missing_desc_rows"),
          F.first(F.when(F.col("Description").isNotNull(), F.col("Description")), ignorenulls=True)
            .alias("existing_description")
      )
      .filter(F.col("missing_desc_rows") > 0)
      .orderBy(F.desc("missing_desc_rows"))
)

summary.show(truncate=False)

+---------+-----------------+-----------------------------------+
|StockCode|missing_desc_rows|existing_description               |
+---------+-----------------+-----------------------------------+
|22139    |12               |RETRO SPOT TEA SET CERAMIC 11 PC   |
|84990    |12               |60 GOLD AND SILVER FAIRY CAKE CASES|
|35965    |11               |FOLKART HEART NAPKIN RINGS         |
|79321    |11               |CHILLI LIGHTS                      |
|22950    |10               |36 DOILIES VINTAGE CHRISTMAS       |
|23084    |10               |RABBIT NIGHT LIGHT                 |
|22084    |9                |PAPER CHAIN KIT EMPIRE             |
|22087    |9                |PAPER BUNTING WHITE LACE           |
|37461    |8                |FUNKY MONKEY MUG                   |
|71477    |8                |short                              |
|22451    |7                |SILK PURSE RUSSIAN DOLL RED        |
|21768    |7                |WALL MOUNTED VINTAGE ORGANISER     |
|22528    

## 4.4. Incoherences 

To be done

## 4.5. Univariate feature analysis

To be done.

## 4.6. Multivariate feature analysis

To be done.

## 4.7. New features exploration

To be done.